# Proyecto Final — Etapa 2: Clasificación de Textos por Década (Mark 2)

**Modelo:** `PlanTL-GOB-ES/roberta-large-bne` — RoBERTa-Large preentrenado sobre +200 GB de texto español incluyendo documentos históricos de la Biblioteca Nacional de España.  
**Licencia:** Apache 2.0 — https://huggingface.co/PlanTL-GOB-ES/roberta-large-bne

**Mejoras sobre Mark 1:**
- **Modelo más grande** (roberta-large vs base)
- **Layer-wise LR Decay (LLRD):** capas inferiores reciben menor LR para preservar conocimiento preentrenado
- **Label smoothing (ε=0.1):** regularización sobre la distribución objetivo
- **Early stopping (patience=3)**
- **Sliding window en inferencia:** textos largos se procesan en ventanas solapadas; el 13% de muestras supera MAX_LEN
- **fp16 (mixed precision):** ~2× velocidad, compatible con PyTorch ≥ 1.9

## 1. Instalación y librerías

In [10]:
!pip install transformers accelerate -q

In [11]:
import os, re, random
from contextlib import nullcontext
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_cosine_schedule_with_warmup,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.optim import AdamW

print(f"PyTorch: {torch.__version__}")

PyTorch: 2.10.0+cu128


## 2. Configuración

In [ ]:
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IS_COLAB  = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ

if IS_KAGGLE:
    DATA_DIR = "/kaggle/input/machine-learning-project"
    OUT_DIR  = "/kaggle/working"
elif IS_COLAB:
    DATA_DIR = "/content"   # subir train.csv y eval.csv directamente aquí
    OUT_DIR  = "/content"
else:
    DATA_DIR = "../../data"
    OUT_DIR  = "../submissions"

TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
EVAL_PATH  = os.path.join(DATA_DIR, "eval.csv")
os.makedirs(OUT_DIR, exist_ok=True)

# RoBERTa-Large entrenado sobre +200 GB de texto español incluyendo docs históricos de la BNE
# Licencia Apache 2.0 — https://huggingface.co/PlanTL-GOB-ES/roberta-large-bne
MODEL_NAME   = "PlanTL-GOB-ES/roberta-large-bne"
MAX_LEN      = 256
STRIDE       = 128
BATCH_SIZE   = 8
GRAD_ACCUM   = 4     # batch efectivo = 32
EPOCHS       = 7
LR           = 1e-5
LLRD         = 0.9
LABEL_SMOOTH = 0.1
PATIENCE     = 3
WARMUP_FRAC  = 0.06
SEED         = 42
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP      = torch.cuda.is_available()

# ── AMP: compatible con PyTorch <2 y ≥2 ──────────────────────────────────────
if USE_AMP:
    try:                                           # PyTorch ≥ 2.0
        _scaler   = torch.amp.GradScaler("cuda")
        _autocast = lambda: torch.amp.autocast("cuda")
    except AttributeError:                         # PyTorch < 2.0
        _scaler   = torch.cuda.amp.GradScaler()
        _autocast = torch.cuda.amp.autocast
else:
    _scaler   = None
    _autocast = nullcontext

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

set_seed(SEED)
print(f"Entorno: {'Kaggle' if IS_KAGGLE else 'Colab' if IS_COLAB else 'Local'}")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}  |  {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

## 3. Datos y limpieza

**Limpieza mínima:** solo caracteres de control y espacios dobles. No se eliminan rasgos ortográficos históricos (eses largas, letras antiguas) porque son **señales útiles** para clasificar la década.

In [13]:
def clean(text):
    text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", "", str(text))
    text = re.sub(r" {2,}", " ", text)
    return text.strip()


train_df = pd.read_csv(TRAIN_PATH)
eval_df  = pd.read_csv(EVAL_PATH)
train_df["text"] = train_df["text"].map(clean)
eval_df["text"]  = eval_df["text"].map(clean)

print(f"Train: {train_df.shape}  |  Eval: {eval_df.shape}")
train_df.head(2)

Train: (31403, 2)  |  Eval: (3490, 2)


,text,decade
0,Honorarias ¡jubiladas. 57 \ndit.ad Pontem de p...,164
1,"gone. Sus amigos , sus clientes, todo \ncuanto...",182


## 4. Preprocesamiento

In [14]:
le = LabelEncoder()
train_df["label"] = le.fit_transform(train_df["decade"])
NUM_CLASSES = len(le.classes_)
print(f"Clases: {NUM_CLASSES}")

train_data, val_data = train_test_split(
    train_df, test_size=0.1, random_state=SEED, stratify=train_df["label"]
)
print(f"Train: {len(train_data)}  |  Val: {len(val_data)}")

Clases: 39
Train: 28262  |  Val: 3141


## 5. Tokenización y Dataset

In [15]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class DecadeDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.texts  = texts.reset_index(drop=True)
        self.labels = labels.reset_index(drop=True) if labels is not None else None

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            str(self.texts[idx]),
            max_length=MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        item = {
            "input_ids":      enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
        }
        if self.labels is not None:
            item["labels"] = torch.tensor(int(self.labels[idx]), dtype=torch.long)
        return item


train_loader = DataLoader(
    DecadeDataset(train_data["text"], train_data["label"]),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=USE_AMP,
)
val_loader = DataLoader(
    DecadeDataset(val_data["text"], val_data["label"]),
    batch_size=BATCH_SIZE * 2, num_workers=2, pin_memory=USE_AMP,
)
print("DataLoaders listos")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

DataLoaders listos


## 6. Modelo y optimizador

**LLRD:** cada capa del encoder recibe `LR × LLRD^(N−i)`, donde N=número de capas. Embeddings reciben el LR más bajo. La cabeza de clasificación recibe el LR completo.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True,
).to(DEVICE)

NUM_ENC_LAYERS = model.config.num_hidden_layers

print(f"Parámetros: {sum(p.numel() for p in model.parameters()):,}")
print(f"Capas encoder: {NUM_ENC_LAYERS}")


# =========================
# Optimizer con Layer-wise LR Decay
# =========================

def build_optimizer(model, lr, llrd, weight_decay=0.01):
    """
    AdamW con Layer-wise Learning Rate Decay.

    - Capas superiores: mayor learning rate.
    - Capas inferiores y embeddings: menor learning rate.
    - Bias y LayerNorm no reciben weight decay.
    """
    no_decay = {"bias", "LayerNorm.weight", "LayerNorm.bias"}
    groups = {}

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue

        # RoBERTa/BERT: roberta.encoder.layer.0... / bert.encoder.layer.0...
        if "encoder.layer." in name:
            layer_id = int(name.split("encoder.layer.")[1].split(".")[0])
            param_lr = lr * (llrd ** (NUM_ENC_LAYERS - 1 - layer_id))
        elif "embeddings" in name:
            param_lr = lr * (llrd ** NUM_ENC_LAYERS)
        else:
            param_lr = lr   # classifier, pooler, etc.

        param_wd = 0.0 if any(nd in name for nd in no_decay) else weight_decay
        key = (round(param_lr, 12), param_wd)
        if key not in groups:
            groups[key] = {"params": [], "lr": param_lr, "weight_decay": param_wd}
        groups[key]["params"].append(param)

    return AdamW(list(groups.values()))


optimizer = build_optimizer(model=model, lr=LR, llrd=LLRD, weight_decay=0.01)

# =========================
# Scheduler
# =========================

steps_per_epoch = max(1, len(train_loader) // GRAD_ACCUM)
total_steps     = steps_per_epoch * EPOCHS
warmup_steps    = max(1, int(WARMUP_FRAC * total_steps))

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

print(f"Grupos de parámetros: {len(optimizer.param_groups)}")
print(f"Steps por época: {steps_per_epoch}  |  Warmup steps: {warmup_steps}  |  Total steps: {total_steps}")

## 7. Entrenamiento

**Label smoothing:** distribución objetivo `(1−ε)·δ_y + ε/(C−1)` con ε=0.1, C=39.

In [17]:
def smooth_loss(logits, labels):
    log_p = F.log_softmax(logits, dim=-1)
    with torch.no_grad():
        tgt = torch.full_like(log_p, LABEL_SMOOTH / (NUM_CLASSES - 1))
        tgt.scatter_(1, labels.unsqueeze(1), 1.0 - LABEL_SMOOTH)
    return -(tgt * log_p).sum(-1).mean()


def train_epoch(loader):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    optimizer.zero_grad()
    for step, batch in enumerate(loader):
        ids  = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        lbls = batch["labels"].to(DEVICE)
        with _autocast():
            logits = model(input_ids=ids, attention_mask=mask).logits
            loss   = smooth_loss(logits, lbls) / GRAD_ACCUM
        if _scaler:
            _scaler.scale(loss).backward()
        else:
            loss.backward()
        if (step + 1) % GRAD_ACCUM == 0 or (step + 1) == len(loader):
            if _scaler:
                _scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            if _scaler:
                _scaler.step(optimizer)
                _scaler.update()
            else:
                optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
        total_loss += loss.item() * GRAD_ACCUM
        correct    += (logits.detach().argmax(-1) == lbls).sum().item()
        n          += len(lbls)
    return total_loss / len(loader), correct / n


def val_epoch(loader):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    with torch.no_grad():
        for batch in loader:
            ids  = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            lbls = batch["labels"].to(DEVICE)
            with _autocast():
                logits = model(input_ids=ids, attention_mask=mask).logits
                loss   = smooth_loss(logits, lbls)
            total_loss += loss.item()
            correct    += (logits.argmax(-1) == lbls).sum().item()
            n          += len(lbls)
    return total_loss / len(loader), correct / n


best_val_acc   = 0.0
patience_count = 0
best_ckpt      = os.path.join(OUT_DIR, "best_bne_large_mark2.pt")

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_epoch(train_loader)
    va_loss, va_acc = val_epoch(val_loader)
    print(f"Epoch {epoch}/{EPOCHS}  train_loss={tr_loss:.4f}  train_acc={tr_acc:.4f}  "
          f"val_loss={va_loss:.4f}  val_acc={va_acc:.4f}")
    if va_acc > best_val_acc:
        best_val_acc   = va_acc
        patience_count = 0
        torch.save(model.state_dict(), best_ckpt)
        print(f"  → Checkpoint guardado (val_acc={va_acc:.4f})")
    else:
        patience_count += 1
        if patience_count >= PATIENCE:
            print(f"  Early stopping en epoch {epoch} (patience={PATIENCE})")
            break

print(f"\nMejor val_acc: {best_val_acc:.4f}")

Epoch 1/7  train_loss=3.4322  train_acc=0.0732  val_loss=3.1317  val_acc=0.1216
  → Checkpoint guardado (val_acc=0.1216)
Epoch 2/7  train_loss=3.0594  train_acc=0.1346  val_loss=2.9743  val_acc=0.1722
  → Checkpoint guardado (val_acc=0.1722)
Epoch 3/7  train_loss=2.9257  train_acc=0.1726  val_loss=2.9118  val_acc=0.1754
  → Checkpoint guardado (val_acc=0.1754)
Epoch 4/7  train_loss=2.8348  train_acc=0.1944  val_loss=2.8412  val_acc=0.1929
  → Checkpoint guardado (val_acc=0.1929)
Epoch 5/7  train_loss=2.7751  train_acc=0.2130  val_loss=2.8261  val_acc=0.1971
  → Checkpoint guardado (val_acc=0.1971)
Epoch 6/7  train_loss=2.7375  train_acc=0.2252  val_loss=2.8013  val_acc=0.2057
  → Checkpoint guardado (val_acc=0.2057)
Epoch 7/7  train_loss=2.7226  train_acc=0.2291  val_loss=2.8019  val_acc=0.2025

Mejor val_acc: 0.2057


## 8. Inferencia con Sliding Window y Envío

Para textos con más tokens que `MAX_LEN` se crean ventanas con paso `STRIDE`. Los logits de cada ventana se **promedian** antes del argmax.

In [18]:
model.load_state_dict(torch.load(best_ckpt, map_location=DEVICE))
model.eval()

CLS_ID = tokenizer.cls_token_id or tokenizer.bos_token_id
SEP_ID = tokenizer.sep_token_id or tokenizer.eos_token_id
PAD_ID = tokenizer.pad_token_id
CHUNK  = MAX_LEN - 2   # tokens de contenido por ventana


@torch.no_grad()
def predict_sliding(texts):
    preds = []
    for text in texts:
        token_ids = tokenizer.encode(str(text), add_special_tokens=False)

        # Crear ventanas solapadas
        windows_ids, windows_mask = [], []
        pos = 0
        while pos < max(len(token_ids), 1):
            chunk    = token_ids[pos: pos + CHUNK]
            seq      = [CLS_ID] + chunk + [SEP_ID]
            pad_len  = MAX_LEN - len(seq)
            windows_ids.append(seq + [PAD_ID] * pad_len)
            windows_mask.append([1] * len(seq) + [0] * pad_len)
            pos += STRIDE
            if pos >= len(token_ids):
                break

        t_ids  = torch.tensor(windows_ids,  dtype=torch.long).to(DEVICE)
        t_mask = torch.tensor(windows_mask, dtype=torch.long).to(DEVICE)
        with _autocast():
            logits = model(input_ids=t_ids, attention_mask=t_mask).logits
        preds.append(logits.mean(0).argmax(-1).item())
    return np.array(preds)


print("Generando predicciones...")
raw_preds  = predict_sliding(eval_df["text"].tolist())
submission = pd.DataFrame({
    "id":     eval_df["id"],
    "answer": le.inverse_transform(raw_preds),
})

sub_path = os.path.join(OUT_DIR, "submission_bne_large_mark2.csv")
submission.to_csv(sub_path, index=False)
print(f"Submission guardado: {sub_path}")
print(f"\nDistribución:\n{submission['answer'].value_counts().sort_index()}")
submission.head(10)

Token indices sequence length is longer than the specified maximum sequence length for this model (538 > 512). Running this sequence through the model will result in indexing errors


Generando predicciones...
Submission guardado: /content/submission_bne_large_mark2.csv

Distribución:
answer
150    111
151    128
152    123
153    106
154    109
155     82
156    122
157     94
158     79
159    143
160     46
161     51
162    149
163     71
164     29
165     33
166      9
167    115
168    137
169     24
170    113
171     93
172    108
173    118
174     66
175    150
176     42
177     96
178    103
179    119
180     84
181    106
182     56
183      5
184     40
185     54
186     44
187    103
188    229
Name: count, dtype: int64


,id,answer
0,0,175
1,1,182
2,2,150
3,3,176
4,4,153
5,5,178
6,6,167
7,7,156
8,8,171
9,9,152
